# Nova: Step 0.2 - Image I/O & Display

Load Start/End images, validate pair, display side-by-side.

In [ ]:
# Install nova in development mode
import sys
sys.path.insert(0, '/content/nova/src')  # Adjust for Colab

from nova.assets.images import (
    load_image,
    validate_image_pair,
    create_side_by_side,
    create_onion_skin,
    image_to_tensor,
    tensor_to_image,
    save_image,
)
from PIL import Image
import torch

print("Nova image utilities loaded")

In [ ]:
# Create sample images for testing (replace with your actual images)
from PIL import Image, ImageDraw

def create_sample_anime_frame(character_pose: str = "standing", size: tuple = (512, 512)) -> Image.Image:
    """Create a simple anime-style test frame."""
    img = Image.new('RGB', size, (240, 245, 255))
    draw = ImageDraw.Draw(img)
    
    w, h = size
    cx, cy = w // 2, h // 2
    
    # Simple character representation
    # Head
    head_r = 60
    draw.ellipse([cx - head_r, cy - 180 - head_r, cx + head_r, cy - 180 + head_r], 
                 fill=(255, 220, 180), outline=(100, 80, 60), width=3)
    
    # Eyes
    draw.ellipse([cx - 20, cy - 190, cx - 10, cy - 180], fill=(50, 50, 100))
    draw.ellipse([cx + 10, cy - 190, cx + 20, cy - 180], fill=(50, 50, 100))
    
    # Body
    if character_pose == "standing":
        # Torso
        draw.rectangle([cx - 50, cy - 120, cx + 50, cy + 20], 
                     fill=(200, 50, 50), outline=(100, 30, 30), width=3)
        # Arms down
        draw.rectangle([cx - 80, cy - 100, cx - 50, cy + 40], 
                     fill=(255, 220, 180), outline=(100, 80, 60), width=3)
        draw.rectangle([cx + 50, cy - 100, cx + 80, cy + 40], 
                     fill=(255, 220, 180), outline=(100, 80, 60), width=3)
        # Legs
        draw.rectangle([cx - 30, cy + 20, cx - 10, cy + 120], 
                     fill=(50, 50, 150), outline=(30, 30, 100), width=3)
        draw.rectangle([cx + 10, cy + 20, cx + 30, cy + 120], 
                     fill=(50, 50, 150), outline=(30, 30, 100), width=3)
    elif character_pose == "hand_raised":
        # Torso
        draw.rectangle([cx - 50, cy - 120, cx + 50, cy + 20], 
                     fill=(200, 50, 50), outline=(100, 30, 30), width=3)
        # Left arm down
        draw.rectangle([cx - 80, cy - 100, cx - 50, cy + 40], 
                     fill=(255, 220, 180), outline=(100, 80, 60), width=3)
        # Right arm raised
        draw.rectangle([cx + 50, cy - 200, cx + 80, cy - 80], 
                     fill=(255, 220, 180), outline=(100, 80, 60), width=3)
        # Legs
        draw.rectangle([cx - 30, cy + 20, cx - 10, cy + 120], 
                     fill=(50, 50, 150), outline=(30, 30, 100), width=3)
        draw.rectangle([cx + 10, cy + 20, cx + 30, cy + 120], 
                     fill=(50, 50, 150), outline=(30, 30, 100), width=3)
    
    return img

# Create test frames
start_frame = create_sample_anime_frame("standing", (512, 512))
end_frame = create_sample_anime_frame("hand_raised", (512, 512))

# Save for testing
start_frame.save("/tmp/start_frame.png")
end_frame.save("/tmp/end_frame.png")

print("Sample frames created and saved to /tmp/")
display(start_frame)
display(end_frame)

In [ ]:
# Load and validate image pair
start_img, end_img = validate_image_pair(
    "/tmp/start_frame.png",
    "/tmp/end_frame.png",
    require_same_size=True,
    require_same_mode=True,
)

print(f"Start: {start_img.size} {start_img.mode}")
print(f"End: {end_img.size} {end_img.mode}")
print("✓ Image pair validated successfully")

In [ ]:
# Display side-by-side comparison
comparison = create_side_by_side(
    [start_img, end_img],
    labels=["Start Frame (Frame 1)", "End Frame (Frame N)"],
    spacing=20,
    label_height=50,
    font_size=18,
)

display(comparison)
comparison.save("/tmp/start_end_comparison.png")
print("✓ Side-by-side comparison saved")

In [ ]:
# Test tensor conversion (for diffusion models)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

start_tensor = image_to_tensor(start_img, normalize=True, device=device)
end_tensor = image_to_tensor(end_img, normalize=True, device=device)

print(f"Start tensor: {start_tensor.shape} [{start_tensor.min():.3f}, {start_tensor.max():.3f}]")
print(f"End tensor: {end_tensor.shape} [{end_tensor.min():.3f}, {end_tensor.max():.3f}]")

# Round-trip test
reconstructed = tensor_to_image(start_tensor, denormalize=True)
print(f"Reconstructed: {reconstructed.size} {reconstructed.mode}")
print("✓ Tensor conversion works")

In [ ]:
# Test onion skinning (prev + current + next ghosted)
# Create a middle frame for testing
middle_frame = create_sample_anime_frame("hand_raised", (512, 512))  # placeholder

onion = create_onion_skin(
    prev_frame=start_img,
    curr_frame=middle_frame,
    next_frame=end_img,
    prev_alpha=0.3,
    next_alpha=0.3,
    curr_alpha=1.0,
)

display(onion)
onion.save("/tmp/onion_skin.png")
print("✓ Onion skinning visualization created")

In [ ]:
# Test save/load round-trip
save_image(start_img, "/tmp/start_saved.png")
reloaded = load_image("/tmp/start_saved.png")
print(f"Saved and reloaded: {reloaded.size} {reloaded.mode}")
print("✓ Save/load works")

## Summary

✅ **Step 0.2 Complete**

**Functions tested:**
- `validate_image_pair()` - Load and validate Start/End images
- `create_side_by_side()` - Side-by-side comparison with labels
- `image_to_tensor()` / `tensor_to_image()` - Diffusion-ready conversion
- `create_onion_skin()` - Onion-skinning for frame inspection
- `save_image()` / `load_image()` - Save/load round-trip

**Ready for Step 0.3:** Configuration Schema (Pydantic models for all user inputs)